# Test4–Test5 concordance

Compare the automated DGIdb-based evaluations in the two `02_analysis.ipynb` notebooks. Results are recomputed from the original CSVs against the same curated truth workbook, across all genes. Run this notebook from its own directory. No human corrections are applied and no workbooks are exported.

**Precision, candidate-based recall, and F1** use only generated PMID–drug–gene candidates, matching the primary scoring in test4 and test5. Truth interactions without a generated prediction do not contribute to that recall denominator. **End-to-end recovery** measures positively predicted truth interactions out of the entire truth set, including missing candidates. DGIdb-unmatched positive predictions are labeled false positives for this comparison; this is not a manual judgment of their biological validity.
This compares test4's v2 run with test5's revised inclusive v3. Failed test5
classifications are audited and excluded from classification agreement and candidate
metrics; failed truth interactions remain unrecovered in the end-to-end denominator.
DGIdb non-membership is a proxy label, not a human judgment of claim validity.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

KEYS = ["pmid", "drug", "gene"]
RUN_FILES = {
    "test4": Path("../test4_curated_dgidb_qwen3_14b/interaction_classifications-rohit-qwen3-14b.csv"),
    "test5": Path("interaction_classifications-rohit-qwen3-14b-v3.csv"),
}
TRUTH_FILE = Path("../../../data/2026-06-29-test2-RohitList-wText.xlsx")
if not RUN_FILES["test5"].exists():
    raise FileNotFoundError("Run test5/01_wags_pretagging.ipynb before this full-curated concordance analysis.")

import hashlib
import json
run_metadata = json.loads(Path("run_metadata.json").read_text())
saved_prompt = Path("v3_system_prompt.text").read_text()
expected_prompt_hash = hashlib.sha256(json.dumps(saved_prompt, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
if run_metadata["prompt_hash"] != expected_prompt_hash:
    raise ValueError("Saved prompt and run metadata do not match")


## Load and validate inputs

Names are matched exactly after uppercasing and trimming whitespace; synonyms are not resolved. Incomplete prediction keys (such as skipped rows) are excluded and counted below. Predictions for complete keys must be Boolean. Identical repeated classifications are consolidated, preserving distinct evidence text; conflicting classifications for the same key raise an error. Missing predictions remain separate from explicit negative predictions.

In [ ]:
def require_columns(df, columns, label):
    missing = set(columns) - set(df.columns)
    if missing:
        raise ValueError(f"{label}: missing required columns {sorted(missing)}")


def normalize_keys(df):
    df = df.copy()
    for column in ["drug", "gene"]:
        df[column] = df[column].astype("string").str.upper().str.strip().replace("", pd.NA)
    df["pmid"] = pd.to_numeric(df["pmid"], errors="raise").astype("Int64")
    return df


def validate_predictions(raw, label):
    require_columns(raw, KEYS + ["interaction", "evidence"], label)
    if label == "test5":
        require_columns(raw, ["prompt_version", "prompt_hash", "classification_status"], label)
        if not raw["classification_status"].isin(["valid", "failed", "skipped"]).all():
            raise ValueError("Unknown test5 classification status")
        if set(raw["prompt_hash"]) != {expected_prompt_hash}:
            raise ValueError("Test5 results do not match the archived assembled prompt")
    normalized = normalize_keys(raw)
    complete = normalized.dropna(subset=KEYS).copy()
    failed = complete["classification_status"].eq("failed") if "classification_status" in complete else pd.Series(False, index=complete.index)
    valid = complete.loc[~failed].copy()
    if label == "test5" and not raw["prompt_version"].eq("v3").all():
        raise ValueError("Test5 results must use prompt v3")
    tokens = valid["interaction"].astype("string").str.strip().str.lower()
    invalid = tokens.isna() | ~tokens.isin(["true", "false"])
    if invalid.any():
        raise ValueError(f"{label}: non-Boolean predictions at rows {valid.index[invalid].tolist()}")
    valid["predicted"] = tokens.map({"true": True, "false": False}).astype("boolean")
    conflicts = valid.groupby(KEYS)["predicted"].nunique()
    if conflicts.gt(1).any():
        raise ValueError(f"{label}: conflicting predictions for {conflicts[conflicts.gt(1)].index.tolist()}")
    evidence = valid.groupby(KEYS, sort=False)["evidence"].agg(
        lambda values: "\n\n".join(dict.fromkeys(values.dropna().astype(str)))
    ).reset_index()
    pairs = valid[KEYS + ["predicted"]].drop_duplicates().merge(
        evidence, on=KEYS, validate="one_to_one"
    )
    audit = {
        "Input rows": len(raw),
        "Incomplete-key rows excluded": len(raw) - len(complete),
        "Failed classifications excluded": int(failed.sum()),
        "Repeated classifications consolidated": len(valid) - len(pairs),
        "Evaluated candidates": len(pairs),
        "PMIDs with candidates": pairs["pmid"].nunique(),
    }
    return pairs, audit


truth_raw = pd.read_excel(TRUTH_FILE)
require_columns(truth_raw, ["pmid", "drug_name", "gene_name", "context"], "truth workbook")
truth_normalized = normalize_keys(truth_raw.rename(columns={"drug_name": "drug", "gene_name": "gene"}))
truth_pairs = truth_normalized.dropna(subset=KEYS)[KEYS + ["context"]].drop_duplicates(KEYS)
truth_set = set(truth_pairs[KEYS].itertuples(index=False, name=None))
if not truth_set:
    raise ValueError("The truth workbook has no complete interaction keys")

runs, audits = {}, {}
for run, path in RUN_FILES.items():
    runs[run], audits[run] = validate_predictions(pd.read_csv(path), run)
display(pd.DataFrame(audits))
print(f"Shared truth set: {len(truth_pairs)} interactions across {truth_pairs['pmid'].nunique()} PMIDs")

## Headline metrics

Rate differences are **test5 minus test4 in percentage points**. Count differences are raw counts.

In [ ]:
def score_candidates(pairs, truth_set):
    is_truth = pd.Series(
        [key in truth_set for key in pairs[KEYS].itertuples(index=False, name=None)],
        index=pairs.index, dtype=bool,
    )
    positive = pairs["predicted"].astype(bool)
    tp = int((is_truth & positive).sum())
    fp = int((~is_truth & positive).sum())
    tn = int((~is_truth & ~positive).sum())
    fn = int((is_truth & ~positive).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "Evaluated candidates": len(pairs), "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "Precision": precision, "Candidate-based recall": recall, "F1": f1,
        "Recovered truth interactions": tp,
        "Total truth interactions": len(truth_set),
        "End-to-end recovery": tp / len(truth_set) if truth_set else np.nan,
    }


metrics = pd.DataFrame({run: score_candidates(pairs, truth_set) for run, pairs in runs.items()})
rate_metrics = ["Precision", "Candidate-based recall", "F1", "End-to-end recovery"]
comparison = metrics.copy()
comparison["Test5 − test4"] = comparison["test5"] - comparison["test4"]
comparison["Difference unit"] = "count"
comparison.loc[rate_metrics, "Test5 − test4"] *= 100
comparison.loc[rate_metrics, "Difference unit"] = "percentage points"
formatted = comparison.copy().astype(object)
for metric in comparison.index:
    for run in RUN_FILES:
        value = comparison.loc[metric, run]
        formatted.loc[metric, run] = f"{value:.2%}" if metric in rate_metrics else f"{value:.0f}"
    formatted.loc[metric, "Test5 − test4"] = f"{comparison.loc[metric, 'Test5 − test4']:+.2f}"
display(formatted)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(rate_metrics))
for offset, run in [(-0.18, "test4"), (0.18, "test5")]:
    bars = ax.bar(x + offset, metrics.loc[rate_metrics, run] * 100, width=0.36, label=run)
    ax.bar_label(bars, fmt="%.2f%%", padding=3, fontsize=9)
ax.set_xticks(x, rate_metrics)
ax.set_ylim(0, 112)
ax.set_ylabel("Percent")
ax.set_title("Classification metrics and end-to-end recovery")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

## Agreement on individual candidates

Candidate overlap uses the union of complete PMID–drug–gene keys. Classification agreement and Cohen’s kappa use **shared candidates only**. Kappa adjusts for agreement expected from each run's positive/negative proportions; it is undefined if there are no shared candidates or expected agreement is 100%. Neither run is treated as ground truth in the agreement matrix.

In [ ]:
aligned = runs["test4"].rename(columns={"predicted": "test4_predicted", "evidence": "test4_evidence"}).merge(
    runs["test5"].rename(columns={"predicted": "test5_predicted", "evidence": "test5_evidence"}),
    on=KEYS, how="outer", indicator="candidate_membership", validate="one_to_one",
)
shared = aligned.loc[aligned["candidate_membership"] == "both"].copy()
shared_count = len(shared)
agreeing = int((shared["test4_predicted"] == shared["test5_predicted"]).sum())
agreement_rate = agreeing / shared_count if shared_count else np.nan
baseline_positive_rate = float(shared["test4_predicted"].mean()) if shared_count else np.nan
v3_positive_rate = float(shared["test5_predicted"].mean()) if shared_count else np.nan
expected_agreement = baseline_positive_rate * v3_positive_rate + (1 - baseline_positive_rate) * (1 - v3_positive_rate)
kappa = (agreement_rate - expected_agreement) / (1 - expected_agreement) if shared_count and not np.isclose(expected_agreement, 1.0) else np.nan

overlap = pd.Series({
    "Candidate union": len(aligned),
    "Shared candidates": shared_count,
    "Test4-only candidates": int((aligned["candidate_membership"] == "left_only").sum()),
    "Test5-only candidates": int((aligned["candidate_membership"] == "right_only").sum()),
    "Candidate intersection / union": shared_count / len(aligned) if len(aligned) else np.nan,
})
display(overlap.to_frame("Value"))
display(pd.Series({
    "Agreeing classifications": agreeing,
    "Disagreeing classifications": shared_count - agreeing,
    "Classification agreement": agreement_rate,
    "Cohen's kappa": kappa,
}).to_frame("Value"))

agreement_matrix = pd.DataFrame(
    [[int(((shared["test4_predicted"] == row) & (shared["test5_predicted"] == col)).sum())
      for col in [True, False]] for row in [True, False]],
    index=pd.Index(["Interaction", "No interaction"], name="Test4"),
    columns=pd.Index(["Interaction", "No interaction"], name="Test5"),
)
display(agreement_matrix)
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(agreement_matrix.to_numpy(), cmap="Blues")
ax.set_xticks([0, 1], agreement_matrix.columns)
ax.set_yticks([0, 1], agreement_matrix.index)
ax.set_xlabel("Test5 prediction")
ax.set_ylabel("Test4 prediction")
ax.set_title("Classification agreement on shared candidates")
for (row, col), count in np.ndenumerate(agreement_matrix.to_numpy()):
    percentage = count / shared_count if shared_count else 0
    ax.text(col, row, f"{count}\n({percentage:.1%})", ha="center", va="center",
            color="white" if count > agreement_matrix.to_numpy().max() * 0.55 else "black")
fig.tight_layout()
plt.show()

unshared_candidates = aligned.loc[aligned["candidate_membership"] != "both"].copy()
if len(unshared_candidates):
    print("Candidates present in only one run (missing predictions are not negatives):")
    display(unshared_candidates)

## Classification disagreements

Truth membership is based on exact normalized keys in the shared DGIdb truth set. Evidence is displayed from both runs.

In [ ]:
disagreements = shared.loc[shared["test4_predicted"] != shared["test5_predicted"]].copy()
disagreements["in_truth"] = [
    key in truth_set for key in disagreements[KEYS].itertuples(index=False, name=None)
]
disagreements = disagreements[KEYS + ["in_truth", "test4_predicted", "test5_predicted", "test4_evidence", "test5_evidence"]].sort_values(KEYS)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(disagreements)

## Agreement on truth interaction recovery

Each truth interaction is recovered only if its exact PMID–drug–gene key has a positive prediction. Missing candidates count as unrecovered here, while their prediction values remain missing in the detail table. A negative prediction also counts as unrecovered.

In [ ]:
recovery = truth_pairs.copy()
for run in RUN_FILES:
    recovery = recovery.merge(
        runs[run][KEYS + ["predicted"]].rename(columns={"predicted": f"{run}_predicted"}),
        on=KEYS, how="left", validate="one_to_one",
    )
    recovery[f"{run}_recovered"] = recovery[f"{run}_predicted"].fillna(False).astype(bool)
baseline_recovered, v3_recovered = recovery["test4_recovered"], recovery["test5_recovered"]
recovery["Recovery category"] = np.select(
    [baseline_recovered & v3_recovered, baseline_recovered & ~v3_recovered, ~baseline_recovered & v3_recovered],
    ["Recovered by both", "Test4 only", "Test5 only"], default="Recovered by neither",
)
category_order = ["Recovered by both", "Test4 only", "Test5 only", "Recovered by neither"]
recovery_summary = recovery["Recovery category"].value_counts().reindex(category_order, fill_value=0).to_frame("Interactions")
recovery_summary["Share of truth set"] = recovery_summary["Interactions"] / len(recovery)
display(recovery_summary)
recovery_agreement = (baseline_recovered == v3_recovered).mean()
print(f"Recovery-status agreement: {recovery_agreement:.1%} across {len(recovery)} truth interactions")

missed_truth = recovery.loc[~(baseline_recovered & v3_recovered)].sort_values(["Recovery category"] + KEYS)
print("Truth interactions missed by at least one run (context included):")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(missed_truth)

assert agreement_matrix.to_numpy().sum() == shared_count
assert recovery_summary["Interactions"].sum() == len(truth_set)
for run in RUN_FILES:
    assert recovery[f"{run}_recovered"].sum() == metrics.loc["Recovered truth interactions", run]

## Interpretation

Compare metric differences alongside classification and recovery agreement. Similar headline scores can hide different decisions on individual candidates; identical recovery rates can also hide different recovered interactions. The paired tables above distinguish these cases.

In [ ]:
display(Markdown(
    f"- Precision: **{metrics.loc['Precision', 'test4']:.2%}** (test4) versus "
    f"**{metrics.loc['Precision', 'test5']:.2%}** (test5), a change of "
    f"**{comparison.loc['Precision', 'Test5 − test4']:+.2f} percentage points**.\n"
    f"- Candidate-based recall: **{metrics.loc['Candidate-based recall', 'test4']:.2%}** versus "
    f"**{metrics.loc['Candidate-based recall', 'test5']:.2%}**.\n"
    f"- End-to-end recovery: **{int(baseline_recovered.sum())}/{len(recovery)}** versus "
    f"**{int(v3_recovered.sum())}/{len(recovery)}**; "
    f"**{int((baseline_recovered & v3_recovered).sum())}** truth interactions recovered by both.\n"
    f"- Shared-candidate classification agreement: **{agreement_rate:.2%}** "
    f"({agreeing}/{shared_count}); **{len(disagreements)}** disagreements; "
    f"Cohen’s kappa **{kappa:.3f}**.\n"
    f"- Truth recovery-status agreement: **{recovery_agreement:.2%}**."
))